# fMRI Reliability Pipeline
- Designed to parametrically analyze fMRI files to identify which pipelines produce the most robust results.

In [ ]:
import os
import pandas as pd
import numpy as np

# Plotting Packages
import seaborn as sns
import matplotlib.pyplot as plt
import configFunctions as config
from nilearn import plotting
from statFunctions import (select_subjects, BASE_CELL_VARS, session_distances, pairwise_distances,
                           complete_design, subject_means, rm_test, rm_anova, print_rm_test, write_rm_test,
                           write_anova_table, raincloud, SECTION)

# Variables
dataFile = 'coordinatesAll_original.csv' # Path to the CSV file containing the coordinates data
paramFile = 'coordinatesAll_param.csv'   # Parameter grid coordinates (main_paramGrid.ipynb), used for subject selection

# Which subjects to analyze (see select_subjects in statFunctions.py):
# 'base_only' - this notebook's criteria; 'param_only' - main_paramGrid.ipynb's criteria;
# 'both' - both criteria, so this notebook and main_paramGrid.ipynb analyze the same subjects.
subjectMode = 'both'

# Toggle nilearn plotting
plot_nilearn = False

# Output directory
output_dir = 'output'
os.makedirs(output_dir, exist_ok=True)

### Required files
- 'coordinatesAll.csv' - generated by serverFunctions, final output of mergeOutputs.sh.
- `coordinatesAll_param.csv` - parameter grid coordinates (see `main_paramGrid.ipynb`), used by `select_subjects` to match subjects across analyses.

In [ ]:
# A configuration function to make output images easier to modify for publication.
config.svg_editing()

In [ ]:
# load data, inspect (subject/session as strings so leading zeros survive and IDs match between files)
df = pd.read_csv(dataFile, dtype={'subject': str, 'session': str})
paramdf = pd.read_csv(paramFile, dtype={'subject': str, 'session': str})
df.head()

### Intermediate clean up
- `select_subjects` (`statFunctions.py`) applies each dataset's inclusion criteria (no missing coordinates, >= 2 sessions in every cell), then keeps the subjects chosen by `subjectMode`.

In [ ]:
# Save the rows with missing coordinates for inspection
rows_with_nan = df[df[['X', 'Y', 'Z']].isna().any(axis=1)]
rows_with_nan.to_csv(os.path.join(output_dir, 'coordinatesAll_NaNRows.csv'), index=False)

# Apply both datasets' inclusion criteria and keep the subjects chosen by subjectMode
df, _ = select_subjects(df, paramdf, mode=subjectMode)
print(f"\n{df['subject'].nunique()} Subjects Remaining...")

In [ ]:
# Mean euclidean distance across sessions for each subject x combination of parameters.
grouping_vars = ['subject', *BASE_CELL_VARS]
csv_out = True  # Toggle: save all pairwise distances to CSV

combodf = session_distances(df, grouping_vars)
combodf = complete_design(combodf, grouping_vars[1:])
combodf.head()

In [ ]:
# Export all pairwise distances for each analysis variant to CSV
if csv_out:
    pairwise_df = pairwise_distances(df, grouping_vars)
    pairwise_csv_path = os.path.join(output_dir, 'pairwise_distances.csv')
    pairwise_df.to_csv(pairwise_csv_path, index=False)
    print(f'Pairwise distances saved to {pairwise_csv_path} ({len(pairwise_df)} rows)')
    display(pairwise_df.head(10))

# Plotting

In [ ]:
# For nilearn brain views, to visualize the output coordinates of processing with the clustMask type mentioned below, must match exactly.
colorDict = {
    'orig': 'red',
    'dilate_5': 'green',
    'erode_1': 'blue'
}

## Nilearn based plotting and visuals

### Single Subject plot

In [ ]:
if plot_nilearn:
    df_subj = df[(df['subject'] == '003')].copy()
    df_subj['colors'] = df_subj['clustMask'].map(colorDict)

    plot_coords = np.array(df_subj[['X','Y','Z']])
    plot_colors = list(df_subj['colors'])
    view = plotting.view_markers(plot_coords, plot_colors, marker_size=5)

    # Viewing options
    # view.open_in_browser()
    view

### Plot the mean coordinates for each subject

In [ ]:
if plot_nilearn:
    df_subj_centroid = df.groupby('subject')[['X', 'Y', 'Z']].mean()
    df_subj['colors'] = df_subj['clustMask'].map(colorDict)

    plot_coords = np.array(df_subj_centroid[['X','Y','Z']])
    # plot_colors = list(df_subj['colors'])
    view = plotting.view_markers(plot_coords, 'red', marker_size=5)
    # view.open_in_browser()
    view

## Raincloud Plot

In [ ]:
targetName = 'sgACC' # For plotting, what does the 'target' mask represent?
clusterRegion = 'dlPFC' # for plotting, what does the 'clusterMask' mask represent?

# Factors tested and plotted (matching df column names)
groupVars = ['sequence_type', 'target', 'clustMask']

# Level order for statistical tests and plotting, and display names for each level
orderDict = {
    'sequence_type': ['se_e2', 'se', 'me'],
    'target': ['seed', 'network'],
    'clustMask': ['erode_1', 'orig', 'dilate_5'],
}
nameDict = {
    'sequence_type': ['Single Echo', 'Multi-Echo', 'Multi-Echo + ICA'],
    'target': ['Seed', 'Network'],
    'clustMask': ['Eroded', 'Original', 'Dilated'],
}
varNameDict = {'sequence_type': 'Sequence Type', 'target': f'{targetName} mask',
               'clustMask': f'{clusterRegion} mask'}

ylabel = 'Mean Pairwise Distance (mm)'

### Statistical tests
- For each factor, average across the other factors so each subject has one value per level, then a paired t-test (2 levels) or RM-ANOVA with Bonferroni post-hoc paired t-tests (3 levels).
- `SD_diff` shows why a larger mean difference can give a higher p-value: t = mean_diff / (SD_diff / sqrt(n)).
- Helpers (`rm_test`, `raincloud`, significance brackets) live in `statFunctions.py`.

In [ ]:
stat_results = {}
agg_dfs = {}

for var in groupVars:
    agg_dfs[var] = subject_means(combodf, var)
    stat_results[var] = rm_test(agg_dfs[var], var, orderDict[var])
    print_rm_test(stat_results[var], varNameDict[var], descriptives=True)
    print()

# 3-Way Repeated Measures ANOVA
- Tests all main effects and interactions in a single model (sequence_type × target × clustMask).
- Uses the full `combodf` without aggregation, so each subject contributes one observation per cell (18 cells total).
- Note: Given the RM nature of the base ANOVA, the 3-way ANOVA will not differ on the individual effects, as each category has its own error term which isn't influenced by the other categories. 
- 2nd Plot is generated for illustrative purpose, to show above reasoning is valid.

In [ ]:
# 3-Way Repeated Measures ANOVA: sequence_type × target × clustMask
full_anova_table = rm_anova(combodf, ['sequence_type', 'target', 'clustMask'], '3-Way Repeated Measures ANOVA')

### Raincloud plot - 3-Way RM ANOVA main effects
- Same layout as the separate analyses, but F and p values reflect the full factorial model.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, var in zip(axes, groupVars):
    # Balanced RM design: each main effect's F/p must equal the one-way test on the subject means plotted
    # as dots (F = t^2 for 2 levels). A mismatch means the design is no longer balanced/complete.
    one_way = stat_results[var]
    one_way_F = one_way['stat'] ** 2 if one_way['test'] == 'ttest' else one_way['stat']
    for name, three_way_val, one_way_val in [('F', full_anova_table.loc[var, 'F Value'], one_way_F),
                                             ('p', full_anova_table.loc[var, 'Pr > F'], one_way['p_val'])]:
        assert np.isclose(three_way_val, one_way_val, rtol=1e-6, atol=0), (
            f'{var}: 3-way {name}={three_way_val:.6g} != one-way {name}={one_way_val:.6g} on subject means')

    # Main-effect F and p from the 3-way RM ANOVA; post-hoc brackets are the same marginal pairwise tests
    result = {**one_way, 'stat_name': 'F',
              'stat': full_anova_table.loc[var, 'F Value'], 'p_val': full_anova_table.loc[var, 'Pr > F']}
    raincloud(ax, agg_dfs[var], var, orderDict[var], nameDict[var], result, varNameDict[var], ylabel)

plt.suptitle('3-Way RM ANOVA: Main Effects', fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'Rainplot_3way.svg'))
plt.show()

### Export statistics to file

In [ ]:
# Save all statistical results to a text file
stats_path = os.path.join(output_dir, 'stats.txt')

with open(stats_path, 'w') as f:
    f.write('Statistical Results Summary\n')
    f.write(f"Data: {dataFile}, subjectMode = '{subjectMode}'\n")
    f.write(f"N subjects: {combodf['subject'].nunique()}\n")
    f.write(SECTION + '\n\n')

    # --- Separate one-way RM ANOVAs / t-tests ---
    f.write('SEPARATE ONE-WAY ANALYSES\n')
    f.write(SECTION + '\n\n')
    for var in groupVars:
        write_rm_test(f, stat_results[var], f'{varNameDict[var]} ({var})')
    f.write(SECTION + '\n\n')

    # --- 3-Way RM ANOVA ---
    write_anova_table(f, full_anova_table, '3-WAY REPEATED MEASURES ANOVA (sequence_type x target x clustMask)')

print(f'Statistics saved to {stats_path}')

### Raincloud plot - half violin, box, and rainplot.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, var in zip(axes, groupVars):
    raincloud(ax, agg_dfs[var], var, orderDict[var], nameDict[var], stat_results[var], varNameDict[var], ylabel)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'Rainplot.svg'))
plt.show()